<a href="https://colab.research.google.com/github/ScholarlyInsane/ml-practice/blob/main/typesof_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Common RNN Architectures Based on Sequence Mapping

RNNs are incredibly versatile because they can handle sequences of varying lengths for both input and output. The way the input and output sequences are mapped defines different common architectures:

### 1. One-to-Many RNN

*   **Description**: This architecture takes a single input and produces a sequence of outputs. The input can be a single data point or a fixed-size vector, and the RNN generates a sequence based on it.
*   **Use Cases**: Image captioning (image input, sequence of words output), music generation (single seed, sequence of notes output), story generation (single prompt, sequence of words output).
*   **Example**: Generating a sequence of characters or words from a single starting character or a concept vector.

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import layers

print("--- One-to-Many RNN Example (Keras) ---")

# Define the model
def create_one_to_many_model(output_sequence_length=10, vocab_size=50):
    model = keras.Sequential([
        layers.Input(shape=(1,)), # Single input feature
        layers.Dense(128, activation='relu'), # Process the single input
        layers.RepeatVector(output_sequence_length), # Repeat the input's representation to match output sequence length
        layers.LSTM(128, return_sequences=True), # LSTM layer to generate sequence
        layers.TimeDistributed(layers.Dense(vocab_size, activation='softmax')) # Output a probability distribution for each step
    ])
    return model

model_one_to_many = create_one_to_many_model()
model_one_to_many.summary()

# Example of dummy input and output shape
dummy_input = np.array([[0.5]]) # A single numerical input
# The model will output a sequence of 10 items, each being a probability distribution over 50 vocabulary items
dummy_output_shape = model_one_to_many.predict(dummy_input).shape
print(f"\nInput shape: {dummy_input.shape}")
print(f"Output shape: {dummy_output_shape}")


--- One-to-Many RNN Example (Keras) ---


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 10, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 10, 128)        │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 10, 50)         │         6,450 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 138,290 (540.20 KB)

 Trainable params: 138,290 (540.20 KB)

 Non-trainable params: 0 (0.00 B)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 346ms/step

Input shape: (1, 1)
Output shape: (1, 10, 50)


### 2. Many-to-Many RNN

*   **Description**: This architecture takes a sequence of inputs and produces a sequence of outputs. There are two main variations:
    *   **Synchronous (or Sequence-to-Sequence with equal length)**: Input and output sequences have the same length, and each output corresponds to a specific input at the same time step (e.g., video frame-by-frame classification).
    *   **Asynchronous (Encoder-Decoder)**: Input and output sequences can have different lengths. An encoder RNN processes the input sequence into a context vector, and a decoder RNN generates the output sequence from that context. This is very common in applications like machine translation.
*   **Use Cases**: Machine translation (sentence in one language, sentence in another), speech recognition (audio sequence, word sequence), video classification (video frames, sequence of labels).
*   **Example**: Translating a sentence from English to French.

In [4]:
print("--- Many-to-Many RNN Example (Keras) ---")

# Define the model (Asynchronous - Encoder-Decoder for machine translation)
def create_many_to_many_model(input_sequence_length=20, output_sequence_length=25, input_vocab_size=100, output_vocab_size=120):
    # Encoder
    encoder_inputs = keras.Input(shape=(input_sequence_length,))
    x = layers.Embedding(input_vocab_size, 128)(encoder_inputs)
    x, state_h, state_c = layers.LSTM(128, return_state=True)(x)
    encoder_states = [state_h, state_c]

    # Decoder
    decoder_inputs = keras.Input(shape=(output_sequence_length,))
    x = layers.Embedding(output_vocab_size, 128)(decoder_inputs)
    decoder_lstm = layers.LSTM(128, return_sequences=True)
    decoder_outputs = decoder_lstm(x, initial_state=encoder_states)
    decoder_dense = layers.Dense(output_vocab_size, activation='softmax')
    decoder_outputs = decoder_dense(decoder_outputs)

    model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)
    return model

model_many_to_many = create_many_to_many_model()
model_many_to_many.summary()

# Example of dummy input and output shapes
dummy_encoder_input = np.random.randint(0, 100, size=(1, 20)) # One input sequence of 20 tokens
dummy_decoder_input = np.random.randint(0, 120, size=(1, 25)) # One target output sequence of 25 tokens

# Note: For actual training, you would shift the target sequence as decoder input
# and predict the actual target sequence.
dummy_output_shape = model_many_to_many.predict([dummy_encoder_input, dummy_decoder_input]).shape
print(f"\nEncoder input shape: {dummy_encoder_input.shape}")
print(f"Decoder input shape: {dummy_decoder_input.shape}")
print(f"Output shape: {dummy_output_shape}")

# Synchronous many-to-many (e.g., for per-step classification)
def create_synchronous_many_to_many_model(sequence_length=10, input_features=5, output_classes=3):
    model = keras.Sequential([
        layers.Input(shape=(sequence_length, input_features)),
        layers.LSTM(64, return_sequences=True), # important for output at each timestep
        layers.TimeDistributed(layers.Dense(output_classes, activation='softmax'))
    ])
    return model

model_sync_many_to_many = create_synchronous_many_to_many_model()
print("\nSynchronous Many-to-Many Model Summary:")
model_sync_many_to_many.summary()

dummy_sync_input = np.random.rand(1, 10, 5)
dummy_sync_output_shape = model_sync_many_to_many.predict(dummy_sync_input).shape
print(f"\nSynchronous Input shape: {dummy_sync_input.shape}")
print(f"Synchronous Output shape: {dummy_sync_output_shape}")


--- Many-to-Many RNN Example (Keras) ---


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 25)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 20, 128)   │     12,800 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 25, 128)   │     15,360 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 128),     │    131,584 │ embedding[0][0]   │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 25, 128)   │    131,584 │ embedding_1[0][0… │
│                     │                   │            │ lstm_2[0][1],     │
│                     │                   │            │ lstm_2[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 25, 120)   │     15,480 │ lstm_3[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 306,808 (1.17 MB)

 Trainable params: 306,808 (1.17 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 10, 64)         │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 10, 3)          │           195 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,115 (70.76 KB)

 Trainable params: 18,115 (70.76 KB)

 Non-trainable params: 0 (0.00 B)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step

Synchronous Input shape: (1, 10, 5)
Synchronous Output shape: (1, 10, 3)


### 3. Many-to-One RNN

*   **Description**: This architecture takes a sequence of inputs and produces a single output. The RNN processes the entire sequence and then outputs a single result, often based on the final hidden state or an aggregation of states.
*   **Use Cases**: Sentiment analysis (sequence of words, single sentiment label), video classification (sequence of frames, single video label), time series forecasting (past series, single future value).
*   **Example**: Classifying the sentiment of a movie review (positive/negative).

In [5]:
print("--- Many-to-One RNN Example (Keras) ---")

# Define the model
def create_many_to_one_model(input_sequence_length=50, input_vocab_size=1000, num_classes=2):
    model = keras.Sequential([
        layers.Input(shape=(input_sequence_length,)), # Sequence of tokens
        layers.Embedding(input_vocab_size, 64), # Embed each token
        layers.LSTM(128), # LSTM layer, return_sequences=False by default for many-to-one
        layers.Dense(num_classes, activation='softmax') # Single output for classification
    ])
    return model

model_many_to_one = create_many_to_one_model()
model_many_to_one.summary()

# Example of dummy input and output shape
dummy_input = np.random.randint(0, 1000, size=(1, 50)) # One input sequence of 50 tokens
dummy_output_shape = model_many_to_one.predict(dummy_input).shape
print(f"\nInput shape: {dummy_input.shape}")
print(f"Output shape: {dummy_output_shape}")


--- Many-to-One RNN Example (Keras) ---


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 50, 64)         │        64,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 163,074 (637.01 KB)

 Trainable params: 163,074 (637.01 KB)

 Non-trainable params: 0 (0.00 B)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step

Input shape: (1, 50)
Output shape: (1, 2)
